In [ ]:
import os, gc, time, csv, warnings
import numpy as np
import pandas as pd
import joblib
import rasterio
from rasterio.windows import Window
import cupy as cp
import cupyx.scipy.ndimage as cpnd
import cuml
from cuml.fil import ForestInference
from sklearn.ensemble import RandomForestRegressor
from tqdm import tqdm

warnings.filterwarnings('ignore')
print(f"cuML version: {cuml.__version__}")

RUN_FOR = 'BGC'   # 'AGC', 'BGC', 'SOC0_30', 'SOC0_100', 'SOC0_200', or 'TECS'

TILE_SIZE       = 3840
NODATA_OUT      = -9999.0
SOC_UPPER_LIMIT = 7000

N_BOOTSTRAP            = 30
BATCH_MODELS           = 30
BOOTSTRAP_N_ESTIMATORS = None
BOOTSTRAP_RANDOM_SEED  = 0
RF_N_JOBS              = -1

FIL_BATCH_SIZE = 500_000
GPU_INFERENCE  = 0
GPU_IDS        = [0, 1]

PIXEL_SIZE_M     = 10
INPAINT_RADIUS_M = 3_000
INPAINT_SIGMA_M  = 800

INPAINT_SIGMA_PX   = max(1, int(round(INPAINT_SIGMA_M / PIXEL_SIZE_M)))
INPAINT_MAX_ITER   = 8
INPAINT_SKIP_RATIO = 0.001
INPAINT_WEIGHT_EPS = cp.float32(1e-4)

ANCHOR_BANDS = {'B2', 'B5', 'B6', 'GRVI', 'MVI', 'NBR', 'MNDWI', 'Clre'}
SOC_TAGS = {'SOC0_30', 'SOC0_100', 'SOC0_200'}

BASE1 = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff'
BASE2 = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff-2'
BASE3 = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff-3'
BASE4 = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff-4'
BASE5 = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff-5'
BASE6 = '/kaggle/input/datasets/antsasarobidyran/mangrove-covariates-geotiff-6'

RASTERS = {
    'BIO1_MAT':   (f'{BASE1}/BIO1.tif',       True,  10.0),
    'BIO12_MAP':  (f'{BASE1}/BIO12.tif',       True,  None),
    'CHM':        (f'{BASE1}/CHM.tif',         False, None),
    'SST_DRY':    (f'{BASE1}/SST_DRY.tif',     False, None),
    'TREECOVER':  (f'{BASE1}/TREECOVER.tif',   False, None),
    'Clre':       (f'{BASE2}/Clre.tif',        False, None),
    'NBR':        (f'{BASE2}/NBR.tif',         False, None),
    'SLOPE':      (f'{BASE2}/SLOPE.tif',       False, None),
    'SST_WET':    (f'{BASE2}/SST_WET.tif',     False, None),
    'TWI':        (f'{BASE5}/TWI.tif',         False, None),
    'B6':         (f'{BASE3}/B6.tif',          False, None),
    'GRVI':       (f'{BASE3}/GRVI.tif',        False, None),
    'MNDWI':      (f'{BASE3}/MNDWI.tif',       False, None),
    'MVI':        (f'{BASE3}/MVI.tif',         False, None),
    'DIST_WATER': (f'{BASE4}/DIST_WATER.tif',  False, None),
    'B2':         (f'{BASE6}/B2.tif',          False, None),
    'B5':         (f'{BASE6}/B5.tif',          False, None),
}

VIF_DIR = '/kaggle/working/VIF'

RESPONSE_CONFIG = {
    'AGC':  {
        'train_csv':  '/kaggle/working/Mangrove_AGC_clean_full.csv',
        'rf_pkl':     '/kaggle/working/ML_Results/AGC/rf_model_AGC.pkl',
        'vif_csv':    f'{VIF_DIR}/vif_final_AGC.csv',
        'response':   'AGC',
        'output_dir': '/kaggle/working/Maps/FIL/AGC',
    },
    'BGC':  {
        'train_csv':  '/kaggle/working/Mangrove_BGC_clean_full.csv',
        'rf_pkl':     '/kaggle/working/ML_Results/BGC/rf_model_BGC.pkl',
        'vif_csv':    f'{VIF_DIR}/vif_final_BGC.csv',
        'response':   'BGC',
        'output_dir': '/kaggle/working/Maps/FIL/BGC',
    },
    'SOC0_30':  {
        'train_csv':  '/kaggle/working/Mangrove_SOC0_30_clean_full.csv',
        'rf_pkl':     '/kaggle/working/ML_Results/SOC0_30/rf_model_SOC0_30.pkl',
        'vif_csv':    f'{VIF_DIR}/vif_final_SOC0_30.csv',
        'response':   'SOC0_30',
        'output_dir': '/kaggle/working/Maps/FIL/SOC0_30',
    },
    'SOC0_100':  {
        'train_csv':  '/kaggle/working/Mangrove_SOC0_100_clean_full.csv',
        'rf_pkl':     '/kaggle/working/ML_Results/SOC0_100/rf_model_SOC0_100.pkl',
        'vif_csv':    f'{VIF_DIR}/vif_final_SOC0_100.csv',
        'response':   'SOC0_100',
        'output_dir': '/kaggle/working/Maps/FIL/SOC0_100',
    },
    'SOC0_200':  {
        'train_csv':  '/kaggle/working/Mangrove_SOC0_200_clean_full.csv',
        'rf_pkl':     '/kaggle/working/ML_Results/SOC0_200/rf_model_SOC0_200.pkl',
        'vif_csv':    f'{VIF_DIR}/vif_final_SOC0_200.csv',
        'response':   'SOC0_200',
        'output_dir': '/kaggle/working/Maps/FIL/SOC0_200',
    },
    'TECS': {
        'train_csv':  '/kaggle/working/Mangrove_TECS_clean_full.csv',
        'rf_pkl':     '/kaggle/working/ML_Results/TECS/rf_model_TECS.pkl',
        'vif_csv':    f'{VIF_DIR}/vif_final_TECS.csv',
        'response':   'TECS',
        'output_dir': '/kaggle/working/Maps/FIL/TECS',
    },
}

assert RUN_FOR in RESPONSE_CONFIG
cfg        = RESPONSE_CONFIG[RUN_FOR]
TRAIN_CSV  = cfg['train_csv']
RF_PKL     = cfg['rf_pkl']
VIF_CSV    = cfg['vif_csv']
RESPONSE   = cfg['response']
OUTPUT_DIR = cfg['output_dir']
os.makedirs(OUTPUT_DIR, exist_ok=True)

_n_gpus = cp.cuda.runtime.getDeviceCount()
if _n_gpus < 2:
    print(f"  WARNING: only {_n_gpus} GPU(s) found — running single-GPU mode")
    GPU_IDS = [0]
GPU_INFERENCE = GPU_IDS[0]
cp.cuda.Device(GPU_INFERENCE).use()

print(f"  GPUs available : {_n_gpus}")
print(f"  Inference GPU  : {GPU_INFERENCE}")
print(f"  Inpaint GPUs   : {GPU_IDS}")


def raster_path(n):           return RASTERS[n][0]
def raster_zero_is_nodata(n): return RASTERS[n][1]
def raster_scale(n):          return RASTERS[n][2]

def load_features(vif_csv):
    features = pd.read_csv(vif_csv)['predictor'].tolist()
    print(f"  Features ({len(features)}): {features}")
    return features

def compute_training_medians(features, train_csv):
    df = pd.read_csv(train_csv).replace(-9999, np.nan).dropna(subset=[RESPONSE])
    if SOC_UPPER_LIMIT and RUN_FOR in SOC_TAGS:
        df = df[df[RESPONSE] < SOC_UPPER_LIMIT].copy()
    out = {}
    for col in features:
        sc   = raster_scale(col)
        vals = df[col].dropna() if col in df.columns else pd.Series(dtype=float)
        if sc:
            vals = vals / sc
        out[col] = float(vals.median()) if len(vals) else 0.0
    return out

def log_mem(label=''):
    try:
        import psutil
        rss = psutil.Process().memory_info().rss / 1e9
        msg = f"  [MEM {label}] CPU {rss:.2f} GB"
        with cp.cuda.Device(GPU_INFERENCE):
            free, total = cp.cuda.Device(GPU_INFERENCE).mem_info
            msg += f" | GPU{GPU_INFERENCE} {(total-free)/1e9:.2f}/{total/1e9:.2f} GB"
        print(msg)
    except Exception:
        pass

def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


def load_original_bundle(rf_pkl_path):
    print(f"\n  Loading original RF PKL: {rf_pkl_path}")
    bundle   = joblib.load(rf_pkl_path)
    rf_model = bundle['model']
    scaler   = bundle['scaler']
    features = bundle['features']

    params = rf_model.get_params()
    for k in ('random_state', 'n_jobs', 'warm_start'):
        params.pop(k, None)
    if BOOTSTRAP_N_ESTIMATORS is not None:
        params['n_estimators'] = BOOTSTRAP_N_ESTIMATORS
    print(f"  Original n_estimators: {rf_model.n_estimators}  "
          f"| Bootstrap n_estimators: {params.get('n_estimators')}")

    sc_mean  = scaler.mean_.astype(np.float32)
    sc_scale = scaler.scale_.astype(np.float32)

    return params, sc_mean, sc_scale, features, scaler


def build_training_matrix(features, train_csv, scaler):
    df = pd.read_csv(train_csv).replace(-9999, np.nan)
    df = df.dropna(subset=[RESPONSE] + features)
    if SOC_UPPER_LIMIT and RUN_FOR in SOC_TAGS:
        df = df[df[RESPONSE] < SOC_UPPER_LIMIT].copy()

    X = df[features].copy()
    for col in features:
        sc = raster_scale(col)
        if sc:
            X[col] = X[col] / sc

    X = X.values.astype(np.float32)
    y = df[RESPONSE].values.astype(np.float32)
    X_scaled = scaler.transform(X).astype(np.float32)

    print(f"  Training matrix: X={X_scaled.shape}, y={y.shape}")
    return X_scaled, y


def build_bootstrap_fil_ensemble(bootstrap_params, X_train_scaled, y_train):
    n_train = X_train_scaled.shape[0]
    fil_models = []

    for b in range(N_BOOTSTRAP):
        t0  = time.time()
        rng = np.random.RandomState(BOOTSTRAP_RANDOM_SEED + b)
        idx = rng.randint(0, n_train, size=n_train)
        X_b, y_b = X_train_scaled[idx], y_train[idx]

        rf_b = RandomForestRegressor(
            **bootstrap_params, random_state=BOOTSTRAP_RANDOM_SEED + b,
            n_jobs=RF_N_JOBS
        )
        rf_b.fit(X_b, y_b)

        with cp.cuda.Device(GPU_INFERENCE):
            fil_b = ForestInference.load_from_sklearn(rf_b)
        fil_models.append(fil_b)

        del rf_b, X_b, y_b
        gc.collect()

        print(f"  [Bootstrap {b+1}/{N_BOOTSTRAP}] fit + FIL-compiled "
              f"({time.time()-t0:.1f}s)")
        if (b + 1) % 10 == 0:
            log_mem(f'after {b+1} bootstrap models')

    log_mem('all bootstrap models compiled')
    return fil_models

def _fil_predict_raw(fil_model, X_gpu):
    raw = fil_model.predict(X_gpu)
    if hasattr(raw, 'values'):
        raw = raw.values
    if isinstance(raw, np.ndarray):
        raw = cp.asarray(raw)
    return raw.ravel().astype(cp.float32)


def _gpu_predict_group(models_group, X_cpu_scaled):
    n_v = X_cpu_scaled.shape[0]
    n_m = len(models_group)
    out = np.empty((n_m, n_v), dtype=np.float32)

    with cp.cuda.Device(GPU_INFERENCE):
        streams = [cp.cuda.Stream(non_blocking=True) for _ in range(n_m)]

        for p0 in range(0, n_v, FIL_BATCH_SIZE):
            p1 = min(p0 + FIL_BATCH_SIZE, n_v)
            X_gpu = cp.asarray(X_cpu_scaled[p0:p1])

            results = [None] * n_m
            for mi, (model, stream) in enumerate(zip(models_group, streams)):
                with stream:
                    results[mi] = _fil_predict_raw(model, X_gpu)

            for stream in streams:
                stream.synchronize()

            for mi in range(n_m):
                out[mi, p0:p1] = cp.asnumpy(results[mi])

            del X_gpu, results
            cp.get_default_memory_pool().free_all_blocks()

    return out


def _inpaint_one_band_on_device(arr_cpu, mask_cpu, fallback, sigma_px, max_iter, device_id):
    n_pixels  = arr_cpu.size
    n_missing = int(mask_cpu.sum())

    if n_missing == 0:
        return arr_cpu
    if n_missing / n_pixels < INPAINT_SKIP_RATIO:
        out = arr_cpu.copy()
        out[mask_cpu] = fallback
        return out

    with cp.cuda.Device(device_id):
        filled  = cp.asarray(arr_cpu,  dtype=cp.float32)
        to_fill = cp.asarray(mask_cpu, dtype=cp.bool_)
        fb      = cp.float32(fallback)
        valid_f = (~to_fill).astype(cp.float32)

        for _ in range(max_iter):
            if not to_fill.any():
                break

            data_f = filled.copy()
            data_f[to_fill] = cp.float32(0.0)

            bd = cpnd.gaussian_filter(data_f.astype(cp.float32), sigma=sigma_px, mode='nearest')
            bw = cpnd.gaussian_filter(valid_f.astype(cp.float32), sigma=sigma_px, mode='nearest')
            del data_f

            can   = to_fill & (bw > INPAINT_WEIGHT_EPS)
            n_can = int(can.sum())
            if n_can == 0:
                del bd, bw, can
                break

            filled[can]  = bd[can] / bw[can]
            to_fill[can] = False
            valid_f[can] = cp.float32(1.0)

            del bd, bw, can

        if to_fill.any():
            filled[to_fill] = fb

        result = cp.asnumpy(filled)
        del filled, to_fill, valid_f
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()

    return result


def inpaint_optional_bands_multigpu(bands, band_missing, optional_names,
                                     train_medians, sigma_px, max_iter, gpu_ids):
    work = []
    for name in optional_names:
        msk = band_missing.get(name)
        if msk is not None and msk.any():
            work.append(name)

    if not work:
        return bands

    n_gpus = len(gpu_ids)

    if n_gpus == 1:
        for name in work:
            bands[name] = _inpaint_one_band_on_device(
                bands[name], band_missing[name],
                train_medians[name], sigma_px, max_iter, gpu_ids[0],
            )
        return bands

    from concurrent.futures import ThreadPoolExecutor, as_completed

    futures = {}
    with ThreadPoolExecutor(max_workers=min(len(work), n_gpus)) as pool:
        for i, name in enumerate(work):
            dev = gpu_ids[i % n_gpus]
            fut = pool.submit(
                _inpaint_one_band_on_device,
                bands[name], band_missing[name],
                train_medians[name], sigma_px, max_iter, dev,
            )
            futures[fut] = name

        for fut in as_completed(futures):
            name        = futures[fut]
            bands[name] = fut.result()

    return bands


def _build_X_tile(rs, cs, features, raster_handles, src_nodata, H, W,
                   train_medians, optional_names):
    re  = min(rs + TILE_SIZE, H);  ce = min(cs + TILE_SIZE, W)
    th  = re - rs;                  tw = ce - cs
    win = Window(cs, rs, tw, th)

    bands        = {}
    valid_anchor = np.ones((th, tw), dtype=bool)
    band_missing = {}

    for name in features:
        src = raster_handles[name]
        arr = src.read(1, window=win).astype(np.float32)
        if arr.shape != (th, tw):
            tmp = np.full((th, tw), np.nan, np.float32)
            tmp[:arr.shape[0], :arr.shape[1]] = arr
            arr = tmp

        nd  = src_nodata[name]
        msk = ~np.isfinite(arr)
        if nd is not None:
            msk |= np.isclose(arr, nd)
        if raster_zero_is_nodata(name):
            msk |= (arr == 0)

        if name in ANCHOR_BANDS:
            valid_anchor &= ~msk
        else:
            band_missing[name] = msk

        bands[name] = arr

    n_v = int(valid_anchor.sum())
    if n_v == 0:
        return None, None, th, tw, win

    bands = inpaint_optional_bands_multigpu(
        bands=bands, band_missing=band_missing,
        optional_names=optional_names, train_medians=train_medians,
        sigma_px=INPAINT_SIGMA_PX, max_iter=INPAINT_MAX_ITER, gpu_ids=GPU_IDS,
    )

    flat = np.where(valid_anchor.ravel())[0].astype(np.int32)
    cols = []
    for name in features:
        col = bands[name].ravel()[flat].copy()
        sc  = raster_scale(name)
        if sc:
            col /= np.float32(sc)
        cols.append(col)

    bands.clear()
    band_missing.clear()

    X = np.column_stack(cols).astype(np.float32)
    return X, flat, th, tw, win


def fil_predict_mean_std_batched(fil_models, sc_mean, sc_scale, features, train_medians):
    print(f'  Bootstrap models : {len(fil_models)}')
    print(f'  Batch size       : {BATCH_MODELS} models predicted concurrently')
    print(f'  FIL chunk        : {FIL_BATCH_SIZE:,} px per group launch')
    print(f'  Inpaint GPUs     : {GPU_IDS}  |  σ={INPAINT_SIGMA_M} m ({INPAINT_SIGMA_PX} px)  '
          f'max_iter={INPAINT_MAX_ITER}')

    optional_names = [f for f in features if f not in ANCHOR_BANDS]

    src_nodata = {}
    with rasterio.open(raster_path(features[0])) as ref:
        profile = ref.profile.copy()
        H, W    = ref.height, ref.width
        crs     = ref.crs
    for name in features:
        with rasterio.open(raster_path(name)) as src:
            src_nodata[name] = src.nodata

    row_starts = list(range(0, H, TILE_SIZE))
    col_starts = list(range(0, W, TILE_SIZE))
    n_cols_t   = len(col_starts)
    tiles = [(ti * n_cols_t + tj + 1, row_starts[ti], col_starts[tj])
             for ti in range(len(row_starts))
             for tj in range(len(col_starts))]
    print(f'  Grid {H}x{W}  |  {len(tiles)} tiles ({TILE_SIZE}px)  |  CRS: {crs}')

    out_prof = profile.copy()
    out_prof.update(dtype='float32', count=1, nodata=NODATA_OUT,
                     compress='lzw', predictor=2,
                     tiled=True, blockxsize=512, blockysize=512, bigtiff='YES')

    PATH_MEAN = os.path.join(OUTPUT_DIR, f'{RUN_FOR}_mean.tif')
    PATH_STD  = os.path.join(OUTPUT_DIR, f'{RUN_FOR}_std.tif')

    raster_handles = {name: rasterio.open(raster_path(name)) for name in features}

    tracking_csv = os.path.join(OUTPUT_DIR, f'tile_tracking_{RUN_FOR}_mean_std.csv')
    trk_fields   = ['tile_idx', 'row_start', 'col_start',
                     'n_valid_pixels', 'inpaint_s', 'infer_s', 'total_s', 'cum_min', 'status']
    trk_file   = open(tracking_csv, 'w', newline='')
    trk_writer = csv.DictWriter(trk_file, fieldnames=trk_fields)
    trk_writer.writeheader()
    trk_file.flush()
    print(f'  Tracking log -> {tracking_csv}')

    log_mem('startup')
    t0 = time.time()
    B  = len(fil_models)
    model_groups = list(chunked(fil_models, BATCH_MODELS))
    print(f'  {len(model_groups)} group(s) of up to {BATCH_MODELS} models each')

    with rasterio.open(PATH_MEAN, 'w', **out_prof) as h_mean, \
         rasterio.open(PATH_STD,  'w', **out_prof) as h_std:

        pbar = tqdm(total=len(tiles), desc='FIL mean+std tiles (batched)',
                    unit='tile', ncols=95)

        for tidx, rs, cs in tiles:
            t_tile   = time.time()
            re = min(rs + TILE_SIZE, H);  ce = min(cs + TILE_SIZE, W)
            th = re - rs;                  tw = ce - cs
            win = Window(cs, rs, tw, th)

            mean_tile = np.full((th, tw), NODATA_OUT, np.float32)
            std_tile  = np.full((th, tw), NODATA_OUT, np.float32)
            n_v       = 0
            t_inp     = 0.0
            infer_s   = 0.0
            status    = 'nodata'

            try:
                t_inp_start = time.time()
                X, flat, _, _, _ = _build_X_tile(
                    rs, cs, features, raster_handles, src_nodata,
                    H, W, train_medians, optional_names)
                t_inp = time.time() - t_inp_start

                if X is not None:
                    n_v  = X.shape[0]
                    X_sc = (X - sc_mean) / sc_scale
                    del X

                    running_sum   = np.zeros(n_v, dtype=np.float64)
                    running_sumsq = np.zeros(n_v, dtype=np.float64)

                    t_i = time.time()
                    for group in model_groups:
                        preds_group = _gpu_predict_group(group, X_sc)
                        running_sum   += preds_group.sum(axis=0, dtype=np.float64)
                        running_sumsq += np.square(preds_group, dtype=np.float64).sum(axis=0)
                        del preds_group
                    infer_s = time.time() - t_i
                    del X_sc

                    mean_v   = running_sum / B
                    variance = (running_sumsq - B * mean_v * mean_v) / max(B - 1, 1)
                    std_vals = np.sqrt(np.clip(variance, 0.0, None)).astype(np.float32)
                    mean_vals = mean_v.astype(np.float32)
                    del running_sum, running_sumsq, mean_v, variance

                    r2, c2 = np.unravel_index(flat, (th, tw))
                    mean_tile[r2, c2] = mean_vals
                    std_tile[r2, c2]  = std_vals
                    del mean_vals, std_vals, r2, c2, flat
                    status = 'ok'

            except Exception as e:
                status = f'error:{e}'

            h_mean.write(mean_tile, 1, window=win)
            h_std.write(std_tile, 1, window=win)
            del mean_tile, std_tile
            gc.collect()

            trk_writer.writerow(dict(
                tile_idx       = tidx,
                row_start      = rs,
                col_start      = cs,
                n_valid_pixels = n_v,
                inpaint_s      = f'{t_inp:.2f}',
                infer_s        = f'{infer_s:.2f}',
                total_s        = f'{time.time()-t_tile:.2f}',
                cum_min        = f'{(time.time()-t0)/60:.2f}',
                status         = status,
            ))
            trk_file.flush()
            pbar.update(1)
            if status not in ('ok', 'nodata'):
                tqdm.write(f"  ERROR tile {tidx}: {status}")

        pbar.close()

    trk_file.close()
    for h in raster_handles.values():
        h.close()
    for dev in GPU_IDS:
        with cp.cuda.Device(dev):
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()

    elapsed = time.time() - t0
    print(f'\nDone in {elapsed/60:.1f} min')
    log_mem('done')
    for p in (PATH_MEAN, PATH_STD):
        if os.path.exists(p):
            print(f'  {os.path.basename(p):<40s}  {os.path.getsize(p)/1e6:.1f} MB')
    print(f'  Tracking -> {tracking_csv}')


def main():
    print(f"  N_BOOTSTRAP: {N_BOOTSTRAP}  |  BATCH_MODELS: {BATCH_MODELS}  |  "
          f"Inference GPU: {GPU_INFERENCE}  |  Chunk: {FIL_BATCH_SIZE:,}")

    print(f"\nLoading features from {VIF_CSV} ...")
    features      = load_features(VIF_CSV)
    train_medians = compute_training_medians(features, TRAIN_CSV)

    bootstrap_params, sc_mean, sc_scale, features, scaler = load_original_bundle(RF_PKL)

    X_train_scaled, y_train = build_training_matrix(features, TRAIN_CSV, scaler)

    print(f"\nFitting + FIL-compiling {N_BOOTSTRAP} bootstrap models "
          f"(kept resident on GPU) ...")
    fil_models = build_bootstrap_fil_ensemble(bootstrap_params, X_train_scaled, y_train)
    del X_train_scaled, y_train
    gc.collect()

    log_mem('before batched prediction')
    fil_predict_mean_std_batched(fil_models, sc_mean, sc_scale, features, train_medians)

    del fil_models
    for dev in GPU_IDS:
        with cp.cuda.Device(dev):
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()


if __name__ == '__main__':
    main()